# Pokémon Wordle:
# Member 2: Random and Filter solvers
A solver is a function that looks at the clues so far and picks the next Pokémon to guess.
This notebook builds two solvers and compares them:
- **Random**: ignores clues, picks any Pokémon not guessed yet (the baseline).
- **Filter**: keeps only the Pokémon that are still possible answers, then guesses one of them.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
DRIVE = '/content/drive/MyDrive/pokemon_wordle'
sys.path.append(DRIVE)

import os
print(os.listdir(DRIVE))   # you should see engine.py, loader.py, data, basic_solvers.py ...

Mounted at /content/drive
['data', 'results', 'notebooks', 'slides', 'report', 'basic_solvers.py', 'entropy_solver.py', 'minimax_solver.py', 'benchmark.py', 'app.py', 'README.txt', 'loader.py', 'engine.py', '__pycache__']


## 1. Write `basic_solvers.py`##

This cell saves our solver code into the shared Drive folder.
- `random_solver`: picks a random Pokémon that has not been guessed.
- `filter_candidates`: a Pokémon stays a candidate only if it would have given
  exactly the same clues we received for every guess so far.
- `filter_solver`: picks one of the remaining candidates.
Key idea: the real answer is never removed, so the candidate list is never empty.

In [ ]:
%%writefile /content/drive/MyDrive/pokemon_wordle/basic_solvers.py

import random

from engine import feedback


def _unguessed(pokemon, history):
    """All Pokemon that have not been guessed yet."""
    guessed = {g["name"] for g, _ in history}
    return [p for p in pokemon if p["name"] not in guessed]


def random_solver(pokemon, history):
    """Baseline: ignores every clue and picks any Pokemon not guessed yet."""
    return random.choice(_unguessed(pokemon, history))


def filter_candidates(pokemon, history):
    """Candidates = Pokemon that are still possible answers.

    A Pokemon stays a candidate only if
      1. it has not been guessed yet, and
      2. for EVERY (guess, clue) in history, pretending it is the secret would
         have produced exactly that clue:  feedback(guess, candidate) == clue
    The real secret always survives, so the list is never empty.
    """
    candidates = []
    for p in _unguessed(pokemon, history):
        if all(feedback(g, p) == clue for g, clue in history):
            candidates.append(p)
    return candidates


def filter_solver(pokemon, history):
    """CSP-style solver: shrink the candidate list, then guess one of them."""
    candidates = filter_candidates(pokemon, history)
    if not candidates:                      # should never happen, but stay safe
        candidates = _unguessed(pokemon, history)
    return random.choice(candidates)

Overwriting /content/drive/MyDrive/pokemon_wordle/basic_solvers.py


### Test 1: candidates shrink and the answer survives
After each guess the candidate list should get smaller,
and the secret Pokémon must always still be in it.

### Test 2: no repeated guesses
A guessed Pokémon cannot be the answer (otherwise the game would be over),
so no solver should ever guess the same Pokémon twice.

### Test 3: benchmark
Each solver plays one game against every possible answer.
We report the average number of guesses (a lost game counts as 9)
and the win rate. Filter should beat Random by a large margin,
because every clue removes impossible Pokémon.

In [ ]:
import importlib, random
import engine, loader, basic_solvers
for m in (engine, loader, basic_solvers):
    importlib.reload(m)
from engine import feedback, play_game, Game, FAIL_SCORE
from loader import load_pokemon
from basic_solvers import random_solver, filter_candidates, filter_solver

mons = load_pokemon()
by_name = {p["name"]: p for p in mons}
print("Loaded", len(mons), "Pokemon")

# --- Test 1: the secret is never removed, and the list shrinks ---
secret = by_name["Gyarados"] if "Gyarados" in by_name else mons[0]
history = []
print("Candidates at start:", len(filter_candidates(mons, history)))
for guess_name in ["Charizard", "Pikachu", "Blaziken"]:
    if guess_name not in by_name or guess_name == secret["name"]:
        continue
    g = by_name[guess_name]
    history.append((g, feedback(g, secret)))
    cands = filter_candidates(mons, history)
    print(f"After guessing {guess_name}: {len(cands)} candidates left")
    assert secret["name"] in [c["name"] for c in cands], "secret was wrongly removed!"
    assert guess_name not in [c["name"] for c in cands], "guessed Pokemon must be removed"

# --- Test 2: neither solver ever repeats a guess ---
for solver in (random_solver, filter_solver):
    for _ in range(20):
        sec = random.choice(mons)
        _, game = play_game(solver, mons, sec)
        names = [g["name"] for g, _ in game.history]
        assert len(names) == len(set(names)), f"{solver.__name__} repeated a guess"
print("No repeated guesses")

# --- Test 3: benchmark both solvers on EVERY possible answer ---
def average_guesses(solver, runs_per_answer=1):
    scores = []
    for secret in mons:
        for _ in range(runs_per_answer):
            score, _ = play_game(solver, mons, secret)
            scores.append(score)
    wins = sum(s < FAIL_SCORE for s in scores)
    return sum(scores) / len(scores), wins / len(scores)

random.seed(42)
for solver in (random_solver, filter_solver):
    avg, win_rate = average_guesses(solver)
    print(f"{solver.__name__:14s} average guesses = {avg:.2f} | win rate = {win_rate:.0%}")

Loaded 386 Pokemon
Candidates at start: 386
After guessing Charizard: 2 candidates left
After guessing Pikachu: 2 candidates left
After guessing Blaziken: 2 candidates left
No repeated guesses
random_solver  average guesses = 8.91 | win rate = 2%
filter_solver  average guesses = 3.62 | win rate = 100%
